In [21]:
import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Сумиране по Айнщайн
Използва се при дълги алгебрични изрази за да се спести писането на суми. Улеснява и генерализира голям набор от операции от линейната алгебра. Представлява следното просто правило

#### По всички повтарящи се индекси имаме сумиране.

Например:

$ a_i B_{ij} = \sum_{i=1}^n a_i B_{ij} = c_j$

Казваме, че сме контрактирали индекса $i$. За да се случи това, размерността на двата обекта, чиито индекси контрактираме, трябва да е еднаква по въпросния индекс. Индексите, по които няма контракция (в горния пример - индекса $j$), се наричат свободни индекси. Във всяко едно равенство, където изпозлваме сумиране по Айнщайн, свободните индекси от двете страни трябва да съвпадат.

Отбеляаваме, че ние работим в Евклидова геометрия и всички индекси са долни. При не Евклидова геометрия, повтарящите се индекси трябва да са долен и горен и в сумата имаме метриката на пространството.

В python използваме функцията einsum от numpy. При нея подаваме първо индексите на обектите разделени със запетайки и индексите на крайния обек след стрелкичка. След това подаваме подред обектите. Трябва размерността на обектите да съвпада по индексите, които контрактираме. За горния пример бихме имали

`np.einsum('i,ij->j', a, B)`


## Задача 1.
Генерирайте произволни 5 мерни вектори $a$ и $b$ и произволни 5х5 матрици $A$ и $B$. Изпозлвайки сумиране по Айнщаин, пресметнете

1. Скараното произведение $a \cdot b$.

2. Произведението $A \cdot b$.

3. Матричното произведение $A \cdot B$.

4. Тензорното произведение $a \otimes b$.

5. Скаларната величина $x = a_j b_k A_{km} B_{jm}$.


In [2]:
np.random.seed(8)

v = np.random.randint(-5, 5, (5, 2))
a = v[:, 0]
b = v[:, 1]

print('a:', a)
print('b:', b)

a: [-2 -4  0 -2 -5]
b: [-1  4  3  3  0]


In [3]:
np.random.seed(8)

v = np.random.randint(-5, 5, (5, 5, 2))
A = v[:, :, 0]
B = v[:, :, 1]

print('A:\n', A)
print('B:\n', B)

A:
 [[-2 -4  0 -2 -5]
 [-4  4 -3  3 -2]
 [ 0  2 -3  4 -4]
 [-1  3  2  1 -1]
 [ 4 -1 -3  1  1]]
B:
 [[-1  4  3  3  0]
 [-2 -3  1  4 -1]
 [ 0  4  1  0  1]
 [ 0 -5 -3  2  3]
 [ 1  0 -3  1  2]]


In [4]:
# Скаларно произведение
np.einsum('i,i', a, b)

np.int32(-20)

In [5]:
# Векторна трансформация
np.einsum('ij,j', A, b)

array([-20,  20,  11,  22, -14], dtype=int32)

In [6]:
# Матрично произведение
np.einsum('ij,jk', A, B)

array([[  5,  14,  11, -31, -12],
       [ -6, -55, -14,   8,  -2],
       [ -8, -38,  -1,  12,  -1],
       [ -6, -10,   2,  10,   0],
       [ -1,   2,   2,  11,   3]], dtype=int32)

In [7]:
# Tензорно произведение
np.einsum('i,j->ij', a, b)

array([[  2,  -8,  -6,  -6,   0],
       [  4, -16, -12, -12,   0],
       [  0,   0,   0,   0,   0],
       [  2,  -8,  -6,  -6,   0],
       [  5, -20, -15, -15,   0]], dtype=int32)

In [8]:
# Скаларна величина 
np.einsum('j,k,km,jm', a, b, A, B)

np.int32(-443)


## Задача 2. 
Заредете данните от файла `data/teaching_methods.csv`. Имплементирайте функция, която извършва едномерна ANOVA приемайки аргументи `(data, predictor, response)`.

$SST = \sum_{\substack{i=1\\j=1}}^{p,n} (y_{ij} - \bar y)^2 = \sum_{\substack{i=1\\j=1}}^{p,n} (y_{ij} - \bar y_i)^2 - \sum_{\substack{i=1\\j=1}}^{p,n}(\bar y_i - \bar y)^2 = SSE + SSR$

$MSE = \frac{SSE}{p(n-1)}$

$MSR = \frac{SSR}{p - 1}$

$p = P(F_{p-1,p(n-1)} > \frac{MSR}{MSE})$

In [9]:
data = pd.read_csv('data/teaching_methods.csv')

print('Shape: ', data.shape)

Shape:  (48, 3)


In [10]:
data.head(5)

,Method,Test,Satisfaction
0,1,3.000,3.001
1,1,2.990,2.994
2,1,3.041,3.032
3,1,3.063,3.059
4,1,3.035,3.026


In [11]:
data.groupby('Method').agg(['count', 'mean'])

Test           Satisfaction          
       count      mean        count      mean
Method                                       
1         16  3.019312           16  3.015125
2         16  3.020437           16  3.013313
3         16  3.027813           16  3.013125

In [12]:
def anova(data, predictor, response):
    y = data[response].to_numpy()
    groups = data[predictor].to_numpy()

    uniques, group_idx, counts = np.unique(groups, return_inverse=True, return_counts=True)

    myi = np.bincount(group_idx, weights=y) / counts
    my = np.mean(y)

    sst = np.sum((y - my)**2)

    ssr = np.sum(counts * (myi - my)**2)
    sse = sst - ssr

    p = len(uniques)
    N = len(y)

    df_r = p - 1
    df_e = N - p

    msr = ssr / df_r
    mse = sse / df_e

    f = msr / mse
    p = 1 - sp.stats.f.cdf(f, df_r, df_e)

    return f, p

f, p = anova(data, 'Method', 'Satisfaction')

print('F statistic: ', f)
print('p value:     ', p)

F statistic:  0.08031841637568052
p value:      0.922954445975542


In [13]:
def anova_pandas(data, predictor, response):
    y = data[response].to_numpy()

    agg = data.groupby(predictor)[response].agg(['count', 'mean'])
    counts = agg['count']

    myi = agg['mean']
    my = y.mean()

    sst = np.sum((y - my)**2)
    
    ssr = np.sum(counts * (myi - my)**2)
    sse = sst - ssr

    N = len(y)
    p = agg.shape[0]

    df_e = (N - p)
    df_r = (p - 1)

    mse = sse / df_e
    msr = ssr / df_r

    f = msr / mse
    p = 1 - sp.stats.f.cdf(f, df_r, df_e)

    return f, p

f, p = anova_pandas(data, 'Method', 'Satisfaction')

print('F statistic: ', f)
print('p value:     ', p)

F statistic:  0.08031841637568052
p value:      0.922954445975542


In [33]:
# Използването на `numpy` е значително по-бързо от операциите в `pandas`
%timeit anova(data, 'Method', 'Satisfaction')
%timeit anova_pandas(data, 'Method', 'Satisfaction')

299 μs ± 10.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
1.47 ms ± 59.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
# Сверяваме с резултата от библиотеката `statsmodels`
model = ols('Satisfaction ~ C(Method)', data=data).fit()

print('F statistic: ', model.fvalue)
print('p value:     ', model.f_pvalue)

F statistic:  0.08031841637570586
p value:      0.9229544459755187



## Задача 3.
Имплементирайте функция, която извършва еднофакторна MANOVA приемайки аргументи `(data, predictor)`.

In [ ]:
def manova(data, predictor):
    response = data.columns[data.columns != predictor]

    x = data[predictor].to_numpy()
    y = data[response].to_numpy()

    uniques, g_idx, counts = np.unique(x, return_inverse=True, return_counts=True)

    # Вектор на общито средни на отклиците: mu -> (1 x r)
    m = y.mean(axis=0)

    # Вектор на средните стойности на групите: mu_i -> (p x r)
    mi = np.zeros((len(uniques), y.shape[1]))
    np.add.at(mi, g_idx, y)
    mi = mi / counts.reshape(-1,1)

    # Отклонение на групите от общото средно за всеки от отклиците: tau_i -> (p x r)
    ti = (mi - m)

    # Вектор на остатъците: e -> (N x r)
    e = (y - mi[g_idx])

    # Within groups SS: SSE -> (r x r), където k и m са са r-мерни индекси
    sse = np.einsum('nk,nm', e, e) 

    # Between groups SS: SSR -> (r x r), където k и m са са r-мерни индекси
    ssr = np.einsum('p,pk,pm', counts, ti, ti)

    w_lambda = np.linalg.det(sse) / np.linalg.det(sse + ssr)

    return w_lambda

w_lambda = manova(data, 'Method')

print('Wilks\' lambda:', w_lambda)

Wilks' lambda: 0.5109441330377944


Моделът на еднофакторна ANOVA се генерализира директно към този за еднофакторна MANOVA като вземем отклика да е многомерен вектор:

Нека имаме един предиктор $X$, който може да заема $p$ на брой стойности $\{X_i\}_{i=1}^p$. Искаме да определим влиянието му над $r$ на брой отклика  $\pmb Y = (Y_1, \dots,Y_r)$. Моделът на MANOVA e:

$$
\pmb Y_{ij} = \pmb\mu + \pmb\tau_i + \pmb\varepsilon_{ij},
$$


където $\pmb\mu$ е средното на всички наблюдения, $\pmb\tau_i =\pmb\mu - \pmb\mu_i$ е отклонението на средното $\pmb\mu_i$ за $i$-тото ниво на предиктора от общото средно и $\pmb\varepsilon_{ij}$ е грешката при измерването на $\pmb Y_{ij}$.
Моделът е екивалентен на този на ANOVA, но всички величини са векторни. Приемаме, че грешките $\pmb\varepsilon_{ij}$ са некорелирани, еднакво нормално разпределени:

$$
\pmb\varepsilon_{ij} \sim \mathcal{N}(0, \sigma),\quad \mathrm{Cov}(\pmb\varepsilon_{ij}, \pmb\varepsilon_{i'j'}) = 0, \forall (i,j)\neq (i',j')
$$

Искаме да проверим дали някое от нивата на предиктора има ефект върху стойнстта на отклика. Тестваме хипотезата:

$$
H_0: \pmb\tau_i = \pmb 0,\ \forall i=1,\dots,k
$$

срещу алтернативата:

$$
H_1: \exists i: \pmb\tau_i \neq \pmb 0
$$

В едномерния случай имахме пълната сума от грешките, която беше скаларна величина. В многомерния случай имаме матрицата на пълната сума на грешките:

$$
SST = \sum_{i=1}^p \sum_{j=1}^{n_i} (\pmb Y_{ij} - \pmb\mu)\otimes(\pmb Y_{ij} - \pmb\mu)
$$

където $n_i$ е броят на измервания над $i$-тото ниво на предиктова. Отново можем да разделим тази сума на сума на отклоненията между групите и сума на отклоненията в групите. За целта разглеждаме:

$$
\begin{align*}
(\pmb Y_{ij} - \pmb\mu)\otimes(\pmb Y_{ij} - \pmb\mu )=& [(\pmb Y_{ij} - \pmb\mu_i) + (\pmb\mu_i - \pmb\mu)]\otimes[(\pmb Y_{ij} - \pmb\mu_i) + (\pmb\mu_i - \pmb\mu)] \\[8pt]
=&(\pmb Y_{ij} - \pmb\mu_i)\otimes(\pmb Y_{ij} - \pmb\mu_i) + (\pmb Y_{ij} - \pmb\mu_i)\otimes( \pmb\mu_i  - \pmb\mu)
+ (\pmb\mu_i  - \pmb\mu)\otimes(\pmb Y_{ij}- \pmb\mu_i) + (\pmb\mu_i - \pmb\mu)\otimes(\pmb\mu_i - \pmb\mu) \\[8pt]
=&(\pmb Y_{ij} - \pmb\mu_i)\otimes(\pmb Y_{ij} - \pmb\mu_i) + (\pmb Y_{ij} - \pmb\mu_i)\otimes \pmb\tau_i
+ \pmb\tau_i \otimes(\pmb Y_{ij}- \pmb\mu_i) + \pmb\tau_i \otimes \pmb\tau_i.
\end{align*}
$$

За средните два члена в сумата от грешките имаме:

$$
\sum_{i=1}^p \sum_{j=1}^{n_i} (\pmb Y_{ij} - \pmb\mu_i)\otimes \pmb\tau_i = \sum_{i=1}^p (\sum_{j=1}^{n_i}  \pmb (Y_{ij} - \pmb\mu_i))\otimes \pmb\tau_i = 0,
$$

където сме използвали линейността на външното произведение и факта, че $\pmb\mu_i$ е средното на $\pmb Y_{ij}$ за определено $i$. Следователно цялата сума става

$$
\begin{align*}
\begin{split}
SST &= \sum_{i=1}^p \sum_{j=1}^{n_i} \Big[ (\pmb Y_{ij} - \pmb\mu_i)\otimes(\pmb Y_{ij} - \pmb\mu_i) +  \pmb\tau_i \otimes \pmb\tau_i \Big]= \\
&= \sum_{i=1}^p \sum_{j=1}^{n_i} (\pmb Y_{ij} - \pmb\mu_i)\otimes(\pmb Y_{ij} - \pmb\mu_i) + \sum_{i=1}^p n_i  (\pmb\tau_i \otimes \pmb\tau_i) =\\
&= SSE + SSR = \pmb W + \pmb B.
\end{split}
\end{align*}
$$

Означили сме $SSE = \pmb W$ (sum of squares Within groups) и $SSR = \pmb B$ (sum of squared Between groups). За тестването на хипотезата $H_0$ ще изпозлваме статистиката Wilks' lambda:

$$
\Lambda*=\frac{|\pmb W|}{|\pmb W + \pmb B|}.
$$

Също като F-статистиката в едномерния случай, тя ни дава оценка за това колко за "разпръснати" резултатите в самите групи сравнено с цялостното разсейване на популацията. Ако тази статистика е достатъчно малка, можем да отхвърлим нулевата хипотеза и да заключим, че предикторът има ефект върху отклика.

Тази статистика е разпределена като F-разпределение, но параметрите на разпределението зависят от броя променливи в отклика и от броя на нивата на предиктора. За малки стойности на тези величини има таблици с точните разпределения. За голям брой променливи или нива и за голем брой измервания, Bartlett дава модификация на $\Lambda*$, която може да се използва вместо оригинала за тестване на хипотезата.

In [40]:
from statsmodels.multivariate.manova import MANOVA

In [52]:
model = MANOVA.from_formula('Test + Satisfaction ~ C(Method)', data=data)

In [53]:
model.mv_test().summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                     Multivariate linear model
====================================================================
                                                                    
--------------------------------------------------------------------
       Intercept          Value    Num DF  Den DF   F Value   Pr > F
--------------------------------------------------------------------
          Wilks' lambda     0.0001 2.0000 44.0000 301093.1399 0.0000
         Pillai's trace     0.9999 2.0000 44.0000 301093.1399 0.0000
 Hotelling-Lawley trace 13686.0518 2.0000 44.0000 301093.1399 0.0000
    Roy's greatest root 13686.0518 2.0000 44.0000 301093.1399 0.0000
--------------------------------------------------------------------
                                                                    
---------------------------------------------------------------------
           C(Method)         Value   Num DF   Den DF  F Value  Pr > F
---------------------------------------------------------------------
              Wilks' lambda  0.5109  4.0000  88.0000   8.7777  0.0000
             Pillai's trace  0.4898  4.0000  90.0000   7.2967  0.0000
     Hotelling-Lawley trace  0.9558  4.0000  51.7778  10.4388  0.0000
        Roy's greatest root  0.9543  2.0000  45.0000  21.4720  0.0000
====================================================================

"""